# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The Croissant schema is available at:

**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**

This dataset provides clinicopathological and molecular data for 77 cancer survivors with second primary colorectal cancer.

In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and discover available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access high-level metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")

## 2. Data Overview
Review the available record sets, their fields, and their `@id`s. All references use `@id` as per Croissant Best Practices.

In [ ]:
# List all record sets in the dataset with their @id and fields
rs_ids = []
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"- Record Set name: {record_set.name}, @id: {record_set.id}")
    rs_ids.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        # Print each field's name and @id
        print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', 'n/a')})")
    print("")

# If available, show the columns for each record set (if it is tabular)
for record_set in dataset.record_sets:
    if hasattr(record_set, 'columns') and record_set.columns:
        print(f"Columns for record set {record_set.name} (@id: {record_set.id}):")
        for column in record_set.columns:
            print(f"    - {column.name} (@id: {column.id}, type: {getattr(column, 'data_type', 'n/a')})")
        print("")

<br/>
### Example: Iterating records from a record set

You can load and peek into records from any record set using its `@id`.

In [ ]:
# Choose a record set - typically the main data table. Adjust if necessary.
main_record_set_id = rs_ids[0]  # If only one, this works. Edit this if needed.

print(f"Showing first 2 records from record set @id: {main_record_set_id}\n")
for idx, record in enumerate(dataset.records(record_set=main_record_set_id)):
    if idx >= 2:
        break
    pprint.pprint(record)

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames for analysis. You can reference and filter column names by their `@id` for a programmatic workflow.

In [ ]:
# Extract all available record sets into dataframes, keyed by record set @id
dataframes = {}

for record_set_id in rs_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows from record set '{record_set_id}'")

# List columns in main table
main_df = dataframes[main_record_set_id]
print("\nColumns in main record set:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate filtering, normalization, and grouping on a numeric field. To follow the guideline of referencing data by `@id`, extract relevant field `@id`s from the overview above.

We'll:
- Select a numeric field (e.g., 'Age at diagnosis (years)') whose `@id` we identify earlier.
- Apply a value threshold filter.
- Normalize the field.
- Optionally group by a categorical field and aggregate.

In [ ]:
# Replace these with actual @id values found above, e.g. 'http://mlcommons.org/croissant/field/age_at_diagnosis'
# For this dataset, we might have a field e.g. 'Age_at_second_CRC_diagnosis', 'Sex', 'Anatomical_location', etc.
df = main_df.copy()

## List available columns for selection (with their @id where available)
print('Columns for EDA:', list(df.columns))

# Pick typical numeric/categorical field names as in clinical tables:
# Below, update with a real column name if possible.
numeric_field = None
for c in df.columns:
    if 'age' in c.lower() and df[c].dtype in ('int64','float64') or df[c].apply(lambda v: str(v).isdigit()).all():
        numeric_field = c
        break

if not numeric_field:
    # Alternatively, pick the first numeric-looking column
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            if (df[c].dtype == 'float64') or (df[c].dtype == 'int64'):
                numeric_field = c
                break
        except Exception:
            continue

if numeric_field:
    print(f'Using numeric field: {numeric_field}')
else:
    raise ValueError('No numeric field found in the data.')

# Filter records age > 50 (example threshold)
threshold = 50
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by categorical field (e.g., 'Sex' or similar, by @id if available)
group_field = None
for c in df.columns:
    if 'sex' in c.lower() or 'gender' in c.lower():
        group_field = c
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nMean {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of our numeric field and optionally stratify by a group/categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized values
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True)
plt.title(f"Distribution of Normalized {numeric_field}")
plt.xlabel(f"{numeric_field} (normalized)")
plt.ylabel('Count')
plt.show()

# If group_field exists and meaningful
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
We have loaded and explored the FAIR^2 dataset using `mlcroissant`, extracted record sets and fields via their `@id`, and applied common data pre-processing and visualization steps. You can now extend this workflow to further statistical or machine learning analyses by leveraging the dataset's rich clinical and molecular variables.